# 🎬 Autonomous Anime Studio: Dual-Engine AI Director (Google Colab & Kaggle)
**100% Autonomous • Zero Hardcoding • Vision Model Director & Self-Healing Supervisor**

This dual-engine studio operates on both **Google Colab (T4 GPU)** and **Kaggle (T4/P100 GPU)**.
When your Colab GPU quota burns out, failover directly to Kaggle (or vice-versa) with zero code modifications.

### Complete Workflow:
1. **Category Selection:** Preserves group rotation (Group A: 5-clip ranking 75-80s, Group B: rotate-phone 30s, Group C: micro-moment 30s).
2. **Dynamic Live Virality Engine:** Discovers what is trending right now (Apple Music/iTunes RSS charts, AniList GraphQL live airing/trending anime, YouTube Shorts live audio velocity). Zero hardcoded titles, songs, or tags.
3. **Candidate Harvesting & Vision Model Scoring:** Local Vision Model (`Qwen2-VL-2B-Instruct` or `SmolVLM` in 4-bit/FP16) inspects raw clips for pure anime authenticity and action intensity.
4. **Cinematic Editing & Compositing:** Minimalist clean HUD, transparent glassmorphism headers, watermark logo, and creator attribution.
5. **Self-Healing Defect Audit Loop:** Model analyzes the rendered master for black frames, silence, or aspect ratio flaws and repairs them autonomously.
6. **SEO & Trending Tag Injection:** Generates high-CTR title, trending tags, unrelated viral hashtags (`#mrbeast`, `#gta6`, `#phonk`), and copyright fair-use disclaimers.
7. **YouTube Release Scheduler:** Restores token from Colab Secrets / Kaggle Secrets and schedules publication.

In [ ]:
# [Cell 1] Environment Detection & Fast Dependency Setup
import os, sys, subprocess

IS_COLAB = 'google.colab' in sys.modules or os.path.exists('/content')
IS_KAGGLE = os.path.exists('/kaggle')

print(f"⚡ Runtime Detected: {'Google Colab' if IS_COLAB else ('Kaggle Notebook' if IS_KAGGLE else 'Local Machine')}")

# 1. Clone or sync latest codebase
if IS_COLAB:
    if not os.path.exists('/content/anime-AUTO'):
        !git clone https://github.com/sp25126/anime-AUTO.git /content/anime-AUTO
    %cd /content/anime-AUTO
    !git pull origin main
elif IS_KAGGLE:
    if not os.path.exists('/kaggle/working/anime-AUTO'):
        !git clone https://github.com/sp25126/anime-AUTO.git /kaggle/working/anime-AUTO
    %cd /kaggle/working/anime-AUTO
    !git pull origin main

# 2. Install FFmpeg & Python Dependencies
!apt-get update -qq && apt-get install -y -qq ffmpeg fonts-dejavu-core > /dev/null
!pip install -q -r requirements.txt
!pip install -q transformers accelerate torch torchvision torchaudio yt-dlp pillow

print("✅ Dependencies successfully installed and environment ready!")

In [ ]:
# [Cell 2] Restore YouTube Token from Cloud Secrets (Zero Laptop Dependency)
import os, json
os.makedirs("config", exist_ok=True)
token_restored = False

# A. Google Colab Secrets (userdata)
try:
    from google.colab import userdata
    token_data = userdata.get('YOUTUBE_TOKEN_JSON')
    if token_data:
        with open("config/youtube_token.json", "w") as f:
            f.write(token_data if isinstance(token_data, str) else json.dumps(token_data))
        token_restored = True
        print("✅ YouTube Token restored from Google Colab Secrets!")
except Exception:
    pass

# B. Kaggle Secrets
if not token_restored:
    try:
        from kaggle_secrets import UserSecretsClient
        token_data = UserSecretsClient().get_secret("YOUTUBE_TOKEN_JSON")
        if token_data:
            with open("config/youtube_token.json", "w") as f:
                f.write(token_data if isinstance(token_data, str) else json.dumps(token_data))
            token_restored = True
            print("✅ YouTube Token restored from Kaggle Secrets!")
    except Exception:
        pass

if not token_restored:
    print("⚠️ Notice: YOUTUBE_TOKEN_JSON secret not found. Production will run in LOCAL PREVIEW mode (no upload).")

In [ ]:
# [Cell 3] Run 100% Dynamic Virality Engine & Vision Director Pipeline
import sys, os
sys.path.append("scripts/engine")

from autonomous_channel_manager import AutonomousChannelManager
from local_vision_director import LocalVisionAIDirector

# 1. Initialize Local Vision Model onto GPU (Qwen2-VL-2B in FP16/4-bit)
director = LocalVisionAIDirector(use_gpu=True)
director.load_local_model()

# 2. Run Autonomous Zero-Touch Production Cycle
# Discovers live trends, scores candidates, harvests, edits, audits, boosts SEO, and schedules release
manager = AutonomousChannelManager()
is_upload = os.path.exists("config/youtube_token.json")

print(f"\n🚀 Starting Autonomous Cycle | Mode: {'LIVE YOUTUBE UPLOAD' if is_upload else 'LOCAL PREVIEW'}")
success = manager.run_zero_touch_cycle(local_only=not is_upload)

if success:
    print("\n==================================================================")
    print("🎉 PRODUCTION SUCCEEDED!")
    print("Master video created, audited, and scheduled without any hardcoded data.")
    print("==================================================================")
else:
    print("\n❌ Production cycle encountered an error. Check supervisor log above.")